In [3]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)

train = pd.read_csv("../data/processed/train.csv")
val = pd.read_csv("../data/processed/validation.csv")

X_train = train.drop(columns=["ADR", "IsCanceled", "ArrivalDate"])
y_train = train["IsCanceled"]
X_val = val.drop(columns=["ADR", "IsCanceled", "ArrivalDate"])
y_val = val["IsCanceled"]

num_cols = X_train.select_dtypes("number").columns
cat_cols = X_train.select_dtypes(exclude="number").columns

prep = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("sc", StandardScaler())]), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=0.01), cat_cols),
])

knn = Pipeline([("prep", prep), ("knn", KNeighborsClassifier(n_neighbors=5))])
knn.fit(X_train, y_train)

pred = knn.predict(X_val)
prob = knn.predict_proba(X_val)[:, 1]
print("accuracy ", accuracy_score(y_val, pred))
print("precision", precision_score(y_val, pred))
print("recall   ", recall_score(y_val, pred))
print("f1       ", f1_score(y_val, pred))
print("roc_auc  ", roc_auc_score(y_val, prob))
print(confusion_matrix(y_val, pred))

accuracy  0.668572044866264
precision 0.4069182389937107
recall    0.2329852358660425
f1        0.2963132585298832
roc_auc   0.5789187993423378
[[5552  943]
 [2130  647]]


In [4]:
from sklearn.model_selection import GridSearchCV
grid = {"knn__n_neighbors": [3, 5, 11, 21, 51],
        "knn__weights": ["uniform", "distance"],
        "knn__p": [1, 2]}

In [5]:
from sklearn.model_selection import PredefinedSplit

X_all = pd.concat([X_train, X_val])
y_all = pd.concat([y_train, y_val])
split = PredefinedSplit([-1] * len(X_train) + [0] * len(X_val))

search = GridSearchCV(knn, grid, scoring="f1", cv=split, n_jobs=-1, refit=False)
search.fit(X_all, y_all)
print(search.best_params_, search.best_score_)

{'knn__n_neighbors': 5, 'knn__p': 1, 'knn__weights': 'uniform'} 0.32202623247399365


c:\Users\hashe\OneDrive\Desktop\CSE 382 - Machine Learning\City-Hotel-Booking\.venv\Lib\site-packages\sklearn\model_selection\_search.py:1237: UserWarning: One or more of the test scores are non-finite: [0.31618294 0.31611343 0.30882353        nan 0.32202623 0.32083145
 0.29631326 0.29631326 0.28851675 0.28776634        nan        nan
 0.25639713 0.25493716 0.25755248 0.25748656 0.23210412 0.22999729
 0.21636463 0.21971366]
  warnings.warn(


In [6]:
res = pd.DataFrame(search.cv_results_)[
    ["param_knn__n_neighbors", "param_knn__weights", "param_knn__p", "mean_test_score"]
].sort_values("mean_test_score", ascending=False)
res

,param_knn__n_neighbors,param_knn__weights,param_knn__p,mean_test_score
4,5,uniform,1,0.322026
5,5,distance,1,0.320831
0,3,uniform,1,0.316183
1,3,distance,1,0.316113
2,3,uniform,2,0.308824
6,5,uniform,2,0.296313
7,5,distance,2,0.296313
8,11,uniform,1,0.288517
9,11,distance,1,0.287766
14,21,uniform,2,0.257552


In [7]:
from sklearn.base import clone

best = {k.replace("knn__", ""): v for k, v in search.best_params_.items()}
final = Pipeline([("prep", clone(prep)), ("knn", KNeighborsClassifier(**best))])

ft = pd.read_csv("../data/processed/final_train.csv")
te = pd.read_csv("../data/processed/test.csv")
drop = ["ADR", "IsCanceled", "ArrivalDate"]

final.fit(ft.drop(columns=drop), ft["IsCanceled"])
pred = final.predict(te.drop(columns=drop))
prob = final.predict_proba(te.drop(columns=drop))[:, 1]
y_te = te["IsCanceled"]

print("accuracy ", accuracy_score(y_te, pred))
print("precision", precision_score(y_te, pred))
print("recall   ", recall_score(y_te, pred))
print("f1       ", f1_score(y_te, pred))
print("roc_auc  ", roc_auc_score(y_te, prob))
print(confusion_matrix(y_te, pred))

accuracy  0.6332857893551536
precision 0.5093643586833144
recall    0.3625530195920016
f1        0.42359882005899707
roc_auc   0.6234217312940412
[[6641 1729]
 [3156 1795]]


In [8]:
import os, joblib
os.makedirs("../outputs/classification", exist_ok=True)
joblib.dump(final, "../outputs/classification/knn.joblib")

['../outputs/classification/knn.joblib']